## Allergen Predictor Demo
This notebook demonstrates the core functionality of the Allergen Predictor. It loads sequence data from the subset datasets, extracts physiochemical properties using Biopython, constructs a labeled dataset, and trains a Random Forest classifier to predict allergenicity.

In [13]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import sys
import os

sys.path.append(os.path.abspath('..'))

from allergen_predictor.data_loader import load_positive_data, load_negative_data
from allergen_predictor.feature_extractor import extract_all_features

print("Libraries and custom modules loaded successfully.")

Libraries and custom modules loaded successfully.


In [14]:
# Load the subset data
# Using the subset files located in the data/ folder
pos_seqs = load_positive_data("../data/subset_pos_file.csv")
neg_seqs = load_negative_data("../data/subset_neg_file.fasta", sample_size=46)

# Extract features and assign binary labels (1 = Allergen, 0 = Non-Allergen)
dataset = []

# Process positive sequences
for seq in pos_seqs:
    features = extract_all_features(seq)
    features['Label'] = 1
    dataset.append(features)

# Process negative sequences
for seq in neg_seqs:
    features = extract_all_features(seq)
    features['Label'] = 0
    dataset.append(features)

# Combine into a data frame
df = pd.DataFrame(dataset)
print(f"Dataset created with {len(df)} total samples.")

# First 5 rows to confirm it works
display(df.head())

Dataset created with 96 total samples.


,Molecular weight,Isoelectric point,Instability index,Aromaticity,Gravy,Label
0,2115.2488,4.050028,76.850000,0.000000,-0.540000,1
1,1211.3877,5.241544,108.216667,0.000000,0.933333,1
2,674.6574,4.085212,-5.816667,0.000000,-1.283333,1
3,1663.9137,6.004892,12.153333,0.133333,0.506667,1
4,1818.9871,9.994937,20.253333,0.200000,-1.200000,1


In [15]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

# Separate features and labels
X = df.drop('Label', axis=1)  # features
y = df['Label']  # labels

# Train the Random Forest Classifier
clf = RandomForestClassifier(random_state=42)
clf.fit(X, y)
print("Random Forest model trained successfully.\n")


# Loads real model trained on the full 21,500 sequences
# Subset is too small to effectively train
production_model_path = "data/allergen_rf_model.pkl"

try:
    prod_clf = joblib.load(production_model_path)
except FileNotFoundError:
    print(f"Error: Could not find the trained model at {production_model_path}.")
    prod_clf = None

if prod_clf:
    # Bad Data: MAGNSRPQAZQPQ
    # Allergen (Ara h 2): RQQWELQGDRRCQSQLER
    # Non-Allergen (Hemoglobin): VHLTPEEKSAVTALWGKVNVDEVG
    
    test_sequence = input("Paste an amino acid sequence to test: ").strip().upper()

    if test_sequence:
        try:
            # Extract features for the new sequence
            test_features = pd.DataFrame([extract_all_features(test_sequence)])
            
            #
            print("\n Extracted Physicochemical Properties:")
            display(test_features.round(4)) 
            
            # Make prediction using the PRODUCTION model (prod_clf)
            prediction = prod_clf.predict(test_features)
            probabilities = prod_clf.predict_proba(test_features)[0] 
            
            # Probabilities[0] is the chance of Non-Allergen, probabilities[1] is the chance of Allergen
            allergen_prob = probabilities[1]
            
            # Format the output based on the prediction
            if prediction[0] == 1:
                print("\nPrediction: Allergen")
                print(f"Confidence: {allergen_prob * 100:.2f}% probability of allergenicity")
            else:
                safe_prob = probabilities[0]
                print("\nPrediction: Non-Allergen")
                print(f"Confidence: {safe_prob * 100:.2f}% probability of being safe")
            
        except Exception as e:
            print(f"\nError processing sequence: {e}")
    else:
        print("No sequence entered.")

Random Forest model trained successfully.



Paste an amino acid sequence to test:  AAHASARQQWELQGD



Prediction: Allergen
Confidence: 98.00% probability of allergenicity

Prediction: Allergen
